# 第三讲 —— 政策函数、时序与人口分布

**异质性主体宏观经济学的计算方法（Computational Methods for Heterogeneous-Agent Macro）**

孙杰

### 环境设置

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Random, LinearAlgebra

# 核心代码

### 参数

In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

In [ ]:
function get_params(;
        β=0.96,
        R=1.04,
        b_grid=loggrid(0.05, 200.0, 400),
        z_grid=[2.7, 2.6, 2.5],
        P_z=[0.95 0.04 0.01;
           0.05 0.9  0.05;
           0.01 0.04 0.95],
    )

    return params = (;β, R,
                    V_shape=(length(b_grid), length(z_grid)),
                    b_grid, z_grid, P_z)
end

In [ ]:
params = get_params(; β=0.90)

### 马尔可夫收入（Markov Income）

人力资本 $z$ 取三个值（高、中、低）—— 这里 $z \in \{2.7, 2.6, 2.5\}$ —— 配以行随机的转移矩阵（transition matrix）$\Pi$，其大部分概率质量集中在对角线上：每个状态都具有持续性。

### 效用函数与对数网格

### 简单的离散网格函数
为了清晰起见，这些函数尽可能简单。在实际应用中，它们还有许多不足之处：例如插值、性能优化等。

In [ ]:
"""
将一个值 `x` 对齐到最近的网格点索引。

"""
snap_idx(grid, x) = argmin(abs.(grid .- x))

In [ ]:
closest_b_ind = snap_idx(params.b_grid, 186.05)

params.b_grid[closest_b_ind]

### 分阶段的后向迭代（Backward iteration in stages）

三个阶段组合成一个作用于二维价值函数（value function）$V \in \mathbb{R}^{N_b \times N_z}$ 上的后向步骤。每个阶段将一个带标签的 $V$ 映射到下一个：

- **阶段 3（消费-储蓄）后向。** 网格搜索 $V^{\mathrm{end}} \mapsto V$。
- **阶段 2（收入）后向。** 查找 $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$，位于 $R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j]$。
- **阶段 1（收入冲击）后向。** 通过 $\Pi^\top$ 的矩阵乘法 $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$。

在期与期之间的边界上，$V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$。

In [ ]:
"""
阶段 1（收入冲击）后向：V_start = V_pre_inc * P_z'。

"""
income_shock_backward(V_post_inc_shock, params) = V_post_inc_shock * params.P_z'

"""
阶段 2（收入）后向：V_pre_inc(b_end, z) = V(R*b_end + z, z)，对齐到最近的网格点。
完全使用广播 —— 没有显式循环。

"""
function income_backward(V_post_inc, params)
    (;R, b_grid, z_grid) = params
    b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')
    return V_pre_inc = V_post_inc[CartesianIndex.(b_post_inc, axes(b_post_inc, 2)')]
end


In [ ]:
params = get_params()

(;R, b_grid, z_grid) = params
b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')

# b_post_inc 的每个元素是阶段 2 结束时的财富（b）索引


In [ ]:
"""
阶段 3（消费-储蓄）后向：在 b_end 的选择上进行网格搜索 V_end → V。

"""
function consumption_saving_backward(V_end, params)
    # 解包 b_grid
    (;b_grid) = params

    # 将 b_grid 移到第 3 维以表示 b_end 的选择
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # 将 V_end 的 b_grid 部分移到第 3 维以表示 b_end 的选择
    # V_end_reshape 形状现在是 [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # 沿 b_end 维（第 3 维）求最大值
    V = maximum(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    # 从 V 中去掉第 3 维并返回
    return dropdims(V; dims=3)
end

In [ ]:
V_end = zeros(params.V_shape)
@show size(V_end)

(;b_grid) = params # b_grid = params.b_grid

b_next_grid = insertdims(b_grid; dims=(1,2))
@show size(b_next_grid)

@show size(V_end)
V_end_reshape = permutedims(insertdims(V_end; dims=3), (3,2,1))
@show size(V_end_reshape)

objective = u.(b_grid .- b_next_grid) .+ V_end_reshape
@show size(objective)

V_pre_consume = maximum(objective; dims=3)
@show size(V_pre_consume)

dropdims(V_pre_consume; dims=3)

In [ ]:
function bellman_operator(V_end, params)
    (;β) = params
    # 阶段 3
    V_post_inc = consumption_saving_backward(V_end, params)

    # 阶段 2
    V_post_inc_shock = income_backward(V_post_inc, params)

    # 阶段 1
    V_start = income_shock_backward(V_post_inc_shock, params)

    # 时间推进
    V_end_new = β .* V_start
    return V_end_new
end

In [ ]:
params = get_params()
V_end = zeros(params.V_shape)
V_end_new = bellman_operator(V_end, params)

### 价值函数迭代（Value Function Iteration，与 L2 相同！）

反复迭代 $V \mapsto \mathcal{T}V$，直到最大绝对值变化降至 `tol` 以下。

In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    V = zeros(params.V_shape)

    Δ, iters = Inf, 0
    while Δ >= tol
        TV = bellman_operator(V, params)
        Δ  = maximum(abs.(TV .- V))
        V  = TV
        iters += 1
        iters > maxiter && error("VFI 在 $maxiter 次迭代内未收敛")
    end

    verbosity >= 1 && println("VFI 在 $iters 次迭代后收敛，误差为 $Δ")
    return V
end

In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)

### 人口就是一个数组（向量、矩阵等）

可以把*人口*（population）看作状态变量上的一个*分布*（distribution）。

如果 $b$ 是唯一的状态变量，那么*人口*就是一个向量 $\lambda$，其中 $\lambda_i$ 表示"拥有财富 $b_i$ 的人数"。

如果 $(b, z)$ 是状态变量，那么人口就是一个矩阵 $\lambda$，其中 $\lambda_{ij}$ 表示"拥有财富 $b_i$ 和人力资本 $z_j$ 的人数"。

### 分阶段的前向迭代（Forward iteration in stages）

我们可以让每个人口矩阵依次经过每个阶段向前演化。

- **阶段 1（收入冲击）前向。** $\lambda \mapsto \lambda\, \Pi$。
- **阶段 2（收入）前向。** 每个 $(b^{\mathrm{end}}, z)$ 单元移动到 $(R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j],\, z)$。
- **阶段 3（消费-储蓄）前向。** 每个 $(b, z)$ 单元移动到 $(b - c^\star(b, z),\, z)$。

两次财富重分箱都对齐到最近的网格点。

In [ ]:
"""
阶段 1 前向：λ_post = λ_pre * P_z。

"""
income_shock_forward(λ, P_z) = λ * P_z

"""
阶段 2 前向：每个 (i_b_end, i_z) 单元移动到 (snap(R*b_end + z), i_z)。
通过广播将目标索引向量化；一次紧凑的散射操作。

"""

function change_λ_wealth(λ, b_inds_new)
    λ_new = zero(λ)
    for old_idx in CartesianIndices(λ)
        λ_new[b_inds_new[old_idx], old_idx[2]] += λ[old_idx]
    end
    return λ_new
end

function income_forward(λ, params)
    (;R, b_grid, z_grid) = params
    dest = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')               # (N_b, N_z)
    return change_λ_wealth(λ, dest)
end

"""
阶段 3 前向：每个 (i_b, i_z) 单元移动到 (c_ind[i_b, i_z], i_z)。
c_ind 正是目标索引 —— 直接散射。

"""
function policy_function(V_end, params)
    # 解包 b_grid
    (;b_grid) = params

    # 将 b_grid 移到第 3 维以表示 b_end 的选择
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # 将 V_end 的 b_grid 部分移到第 3 维以表示 b_end 的选择
    # V_end_reshape 形状现在是 [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # 沿 b_end 维（第 3 维）求 argmax
    policy_idxs = argmax(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    return [idx[3] for idx in policy_idxs[:,:,1]]
end

consumption_saving_forward(λ, b_inds_new, params) = change_λ_wealth(λ, b_inds_new)

"""
复合算子 T* 的一次前向步骤：
λ → 收入冲击 → 收入 → 消费储蓄。

"""
function simulate_population_forward(λ, b_inds_new, params)
    # 阶段 1
    λ = income_shock_forward(λ, params.P_z)

    # 阶段 2
    λ = income_forward(λ, params)

    # 阶段 3
    λ = consumption_saving_forward(λ, b_inds_new, params)

    return λ
end


In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)

# 猜测 λ
λ_start = fill(1/length(params.V_shape), params.V_shape)
@show sum(λ_start)


In [ ]:
function find_steady_state_population(V_end, params; tol=1e-5, maxiter=10_000, verbosity=0)

    b_inds_new = policy_function(V_end, params)

    λ = fill(1/length(params.V_shape), params.V_shape) # 让家庭在所有网格点上均匀分布作为起点

    Δ, iters = Inf, 0
    while Δ >= tol
        λ_new = simulate_population_forward(λ, b_inds_new, params)
        Δ = maximum(abs.(λ_new .- λ))
        λ = λ_new
        iters += 1
        iters > maxiter && error("稳态 λ 在 $maxiter 次迭代内未收敛")
    end

    verbosity >= 1 && println("稳态 λ 在 $iters 次迭代后收敛，误差为 $Δ")
    return λ
end

In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)
λ_steady_state = find_steady_state_population(V_end, params; verbosity=1)

In [ ]:
plot(vec(λ_steady_state[:,1]))

## 演示

设置参数，求解价值函数，读取政策函数（policy function），然后运行两种模拟器。

In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)

λ_start = fill(1/length(params.V_shape), params.V_shape)
λ_next = simulate_population_forward(λ_start, b_inds_new, params)
;

### 绘制 $V$ 和 $c^\star$

In [ ]:
(;b_grid, z_grid) = params
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)

# 将索引向量 b_inds_new 转回消费值以便绘图：
c_pol = b_grid .- b_grid[b_inds_new]

p1 = plot(b_grid, V_end; lw=2, xlabel="wealth b", ylabel="V(b)",
          label=["high z" "medium z" "low z"], legend=:bottomright)
p2 = plot(b_grid, c_pol; lw=2, xlabel="wealth b", ylabel="c*(b)",
          label="c*", legend=:bottomright)
plot!(p2, b_grid, b_grid; ls=:dash, lw=1, label="45°")
plot(p1, p2; layout=(1, 2), size=(1000, 320))


### 模拟单个家庭

让一个家庭随时间向前演化。财富在高 $z$ 期间上升，在人力资本切换到较低状态时下降。

In [ ]:
"""
给定当前状态 i，从行随机矩阵 P_z 的马尔可夫链中抽取下一个状态。

"""
sample_markov(P_z, i) = findfirst(rand() .<= cumsum(P_z[i, :]))

"""
使用政策索引矩阵 `c_inds` 让一个家庭向前演化 T 期。
每期依次应用阶段 1（z 上的马尔可夫）、阶段 2（手头现金 b = R·b_end + z）、
阶段 3（储蓄 = 在 (b, z) 处的政策查找）。记录的 `i_b_path[t]` 是 t 时刻
的手头现金（z 实现后的财富）。

"""
function simulate_one(c_inds, params; T=200, b_ind_init=1, z_ind_init=1)
    (;R, z_grid, P_z, b_grid) = params

    # 跨迭代传递的状态：期末储蓄索引 `b_ind`（b_end）和当前人力资本
    # 状态 `z_ind`。每期首先抽取新的收入冲击，与后向 Bellman 的时序匹配。
    b_ind, z_ind = b_ind_init, z_ind_init

    i_b_path, i_z_path = zeros(Int, T), zeros(Int, T)
    for t in 1:T
        # 阶段 1：收入冲击 —— 抽取下一个 z 状态。
        z_ind = sample_markov(P_z, z_ind)

        # 阶段 2：实现收入，组合手头现金 b = R·b_end + z。
        b_cash_ind = snap_idx(b_grid, R * b_grid[b_ind] + z_grid[z_ind])

        # 阶段 3：储蓄 = 在 (手头现金, z) 处的政策。
        b_ind = c_inds[b_cash_ind, z_ind]

        i_b_path[t], i_z_path[t] = b_cash_ind, z_ind
    end
    return (;i_b_path, i_z_path)
end

In [ ]:
(;b_grid, z_grid) = params
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)


Random.seed!(1)
# 从最接近财富 1.0 的网格点开始，处于高 z 状态
sim = simulate_one(b_inds_new, params; T=300,
                   b_ind_init=argmin(abs.(b_grid .- 1.0)), z_ind_init=1)

t_axis = 1:length(sim.i_b_path)
p1 = plot(t_axis, b_grid[sim.i_b_path]; lw=1.5,
          xlabel="t", ylabel="wealth b_t",
          label="b_t", legend=:topright)
p2 = plot(t_axis, [z_grid[z] for z in sim.i_z_path]; lw=1.5,
          xlabel="t", ylabel="human capital z_t",
          label="z_t", legend=:topright, seriestype=:steppost)
plot(p1, p2; layout=(2, 1), size=(820, 420))


In [ ]:
# 让所有人从财富 ≈ 1.0、高 z 状态开始
λ_0 = zeros(length(b_grid), length(z_grid))
i_0 = argmin(abs.(b_grid .- 1.0))
λ_0[i_0, 1] = 1.0

λ_path = simulate_population(c_ind, params, λ_0; T=200)
;


In [ ]:
# 四个时刻的财富边际分布快照
function wealth_marginal(λ)
    return vec(sum(λ; dims=2))
end

snapshots = [1, 5, 25, 200]
p = plot(xlabel="wealth b", ylabel="density", legend=:topright)
for t in snapshots
    plot!(p, b_grid, wealth_marginal(λ_path[t]); lw=1.8, label="t = $(t-1)")
end
plot(p; size=(820, 320), xscale=:log10)


### 从 $V$ 和 $c^\star$ 计算横截面总量

每个 $t$ 时刻的总福利 $\bar V$、总消费 $\bar C$、总财富 $\bar K$ —— 都是与财富边际分布的内积。

In [ ]:
# 与财富边际分布的内积
V_bar = [dot(V,      wealth_marginal(λ)) for λ in λ_path]
C_bar = [dot(c_pol,  wealth_marginal(λ)) for λ in λ_path]
K_bar = [dot(b_grid, wealth_marginal(λ)) for λ in λ_path]

t_axis = 0:length(λ_path)-1
p1 = plot(t_axis, V_bar; lw=1.8, xlabel="t", ylabel="V̄_t", label="V̄")
p2 = plot(t_axis, C_bar; lw=1.8, xlabel="t", ylabel="C̄_t", label="C̄")
p3 = plot(t_axis, K_bar; lw=1.8, xlabel="t", ylabel="K̄_t", label="K̄")
plot(p1, p2, p3; layout=(1, 3), size=(1000, 280))


# 逐步讲解

### 用 `argmax` 求政策函数的分解

In [ ]:
# 在一维 V 切片上重做 L02 的确定性 Bellman（常数 z = 1.0），
# 以展示 argmax 在 (b, b') 矩阵上返回什么。
z_const = 1.0
b_end = (b_grid' .- z_const) ./ params.R
M     = u.(b_grid .- b_end) .+ params.β .* V'
size(M)


In [ ]:
# 沿列方向求 `argmax`：对每一行（每个 b），哪一列（b'）最优？
idx = argmax(M; dims=2)
idx[1:5]      # 前五个 b 的 CartesianIndex(i, j*)


In [ ]:
# 仅提取列索引 —— 这就是所选的 b' 索引
j_star = vec(getindex.(idx, 2))
j_star[1:5]


In [ ]:
# 每个 b 所选的 b' 索引正是 policy_function 返回的结果
c_ind_check = vec(getindex.(idx, 2))
maximum(abs.(c_ind_check .- c_ind))   # 应当为 0


### 沿单一维度的马尔可夫演化即矩阵乘法

只取 $z$ 的边际分布 $\lambda_z$（长度为 3）。马尔可夫一步即 $\lambda_z \cdot \Pi$。经过多步后 $z$ 的份额停止变化 —— 这就是 $\Pi$ 所蕴含的各人力资本状态的长期份额。

In [ ]:
# 起点：所有人都处于高 z 状态
λ_z_0 = [1.0  0.0  0.0]                         # 行向量

# 一步
λ_z_1 = λ_z_0 * params.P_z

# 向前迭代多次；z 份额收敛到其长期值
λ_z = let λ = λ_z_0
    for t in 1:200
        λ = λ * params.P_z
    end
    λ
end

(long_run_high_z=λ_z[1], long_run_medium_z=λ_z[2], long_run_low_z=λ_z[3])


### 追踪一个家庭通过三个阶段

选取单个 $(i_b, i_z)$ 单元，先抽取一步马尔可夫，然后应用阶段 2--3（手头现金的组装与索引查找），看看其质量最终落在何处。

In [ ]:
# 追踪一个 (i_b, i_z) 单元通过阶段 1、2、3
i_b, i_z = 50, 1

# 阶段 1：一次马尔可夫抽取推进 z 状态
Random.seed!(42)
i_z_new = sample_markov(params.P_z, i_z)

# 阶段 2：使用实现的 z 组装手头现金 b = R·b_end + z，对齐到网格
b_cash    = params.R * b_grid[i_b] + z_grid[i_z_new]
i_b_cash  = snap_idx(b_grid, b_cash)

# 阶段 3：储蓄 = 在 (手头现金, z) 处的政策查找
i_b_new = b_inds_new[i_b_cash, i_z_new]
b_new   = b_grid[i_b_new]

(i_b=i_b, i_z=i_z, i_z_new=i_z_new,
 b_cash=b_cash, i_b_cash=i_b_cash,
 i_b_new=i_b_new, b_new=b_new)

多次迭代前向步骤。财富分布稳定到一个长期形状；总量可通过与 $V$、$c^\star$ 和 $b$ 的内积读出。

In [ ]:
# 让所有人从相同的财富和 z 出发，然后多次应用前向步骤 ——
# 一旦迭代足够多次，分布将不再变化。
λ_long_run = let λ = zeros(length(params.b_grid), length(params.z_grid))
    i_0 = argmin(abs.(params.b_grid .- 1.0))
    λ[i_0, 1] = 1.0
    for t in 1:400
        λ = forward_step(λ, c_ind_2d, params)
    end
    λ
end

# 长期总量
λ_z = vec(sum(λ_long_run; dims=1))         # z 边际分布
λ_b = vec(sum(λ_long_run; dims=2))         # 财富边际分布

(high_z_share=λ_z[1],
 mean_wealth=dot(params.b_grid, λ_b),
 mean_V=sum(V_2d .* λ_long_run))
